import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import itertools
import warnings

In [2]:
import pandas as pd

In [3]:
# -----------------------------
# 1️⃣ Load NGFS NiGEM Excel
# -----------------------------
ngfs_file = "NiGEM_data.xlsx"
ngfs = pd.read_excel(ngfs_file)

# -----------------------------
# 2️⃣ Filter for US & relevant variables
# -----------------------------
ngfs_us = ngfs[ngfs['Region'] == 'NiGEM NGFS v1.24.2|United States']
ngfs_us = ngfs_us[ngfs_us['Variable'].str.contains('combined', case=False, na=False)]

# -----------------------------
# 3️⃣ Melt annual columns into long format
# -----------------------------
year_cols = [c for c in ngfs_us.columns if c.isdigit()]
ngfs_long = ngfs_us.melt(id_vars=['Model','Scenario','Region','Variable','Unit'], 
                         value_vars=year_cols,
                         var_name='Year', value_name='Value')

# Convert Year to datetime
ngfs_long['Year'] = pd.to_datetime(ngfs_long['Year'].astype(str) + '-12-31')

# -----------------------------
# 4️⃣ Pivot: Scenario + Variable as columns
# -----------------------------
ngfs_pivot = ngfs_long.pivot_table(index='Year', 
                                   columns=['Scenario','Variable'], 
                                   values='Value')

# Optional: flatten column MultiIndex
ngfs_pivot.columns = [f"{sc}_{var}" for sc, var in ngfs_pivot.columns]

# -----------------------------
# 5️⃣ Interpolate to quarterly
# -----------------------------
ngfs_quarterly = ngfs_pivot.resample('QE').interpolate(method='linear') 

# Quick check
print(ngfs_quarterly.head(10))
print("\nColumns:", ngfs_quarterly.columns.tolist())

            Below 2°C_Carbon pricing ; $ per Tn CO2(combined)  \
Year                                                            
2022-12-31                                           0.000000   
2023-03-31                                           1.908148   
2023-06-30                                           3.816295   
2023-09-30                                           5.724443   
2023-12-31                                           7.632591   
2024-03-31                                           8.359504   
2024-06-30                                           9.086418   
2024-09-30                                           9.813331   
2024-12-31                                          10.540245   
2025-03-31                                          11.267158   

            Below 2°C_Central bank Intervention rate (policy interest rate) ; %(combined)  \
Year                                                                                        
2022-12-31                       

In [1]:
# =========================================================
# MULTIMODAL LOAN GROWTH MODEL (AUTO-ARIMA OPTIMIZED)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
import itertools
import warnings

warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# 1. CLEAN FRED DATA
# ---------------------------------------------------------
def clean_fred_data(file_path, value_col_name):
    try:
        df = pd.read_csv(file_path)
        date_col = [c for c in df.columns if 'date' in c.lower()][0]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.select_dtypes(include=[np.number])
        df_q = df.resample('QE').mean()
        if not df_q.empty:
            df_q.columns = [value_col_name]
        return df_q
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return pd.DataFrame()

# Load data
df_loans = clean_fred_data('BUSLOANS.csv', 'Loans_Billions')
df_unemp = clean_fred_data('UNEMPLOYMENT.csv', 'Unemployment_Rate')
df_gdp = clean_fred_data('GDPC1.csv', 'GDP')
df_rates = clean_fred_data('FEDFUNDS.csv', 'Rates')

# ---------------------------------------------------------
# 2. MERGE + TARGET VARIABLE
# ---------------------------------------------------------
df_history = pd.concat([
    df_loans, df_unemp, df_gdp, df_rates
], axis=1).dropna()

df_history['Loan_Growth'] = np.log(df_history['Loans_Billions']).diff(4) * 100
df_history['GDP'] = np.log(df_history['GDP']).diff(4) * 100

# ---------------------------------------------------------
# 3. COVID DUMMY
# ---------------------------------------------------------
df_history['COVID_Dummy'] = 0
df_history.loc['2020-03-01':'2021-06-30', 'COVID_Dummy'] = 1

# ---------------------------------------------------------
# 4. FEATURE ENGINEERING
# ---------------------------------------------------------
df_history['Unemployment_Lag1'] = df_history['Unemployment_Rate'].shift(1)
df_history['GDP_Lag1'] = df_history['GDP'].shift(1)
df_history['Rates_Lag1'] = df_history['Rates'].shift(1)

df_train = df_history.dropna()

y = df_train['Loan_Growth']
X = df_train[['Unemployment_Lag1', 'GDP_Lag1', 'COVID_Dummy']]

# ---------------------------------------------------------
# 5. ENHANCED MODEL SELECTION (AIC + BIC + SIGNIFICANCE)
# ---------------------------------------------------------
def evaluate_arima_models(y, X, seasonal_period=4,
                         p_range=range(0,4),
                         d_range=range(0,2),
                         q_range=range(0,4),
                         P_range=range(0,2),
                         D_range=range(0,2),
                         Q_range=range(0,2),
                         significance_level=0.05): # signifance level should be .05 not .1

    results_list = []

    print("🔍 Running FULL model evaluation...\n")

    for order in itertools.product(p_range, d_range, q_range):
        for seasonal in itertools.product(P_range, D_range, Q_range):

            seasonal_order = (seasonal[0], seasonal[1], seasonal[2], seasonal_period)

            try:
                model = ARIMA(
                    y,
                    exog=X,
                    order=order,
                    seasonal_order=seasonal_order,
                    enforce_stationarity=True,
                    enforce_invertibility=False
                )

                res = model.fit()

                pvals = res.pvalues
                exog_pvals = {var: pvals.get(var, np.nan) for var in X.columns}

                sig_vars = [k for k, v in exog_pvals.items() if v < significance_level]
                num_sig = len(sig_vars)

                results_list.append({
                    'order': order,
                    'seasonal_order': seasonal_order,
                    'aic': res.aic,
                    'bic': res.bic,
                    'num_significant': num_sig,
                    'significant_vars': sig_vars,
                    'all_significant': num_sig == len(X.columns),
                    'model_obj': res
                })

            except:
                continue

    results_df = pd.DataFrame(results_list)

    # Best models
    best_aic = results_df.sort_values('aic').iloc[0]
    best_bic = results_df.sort_values('bic').iloc[0]

    all_sig_models = results_df[results_df['all_significant'] == True]
    best_all_sig = all_sig_models.sort_values('aic').iloc[0] if not all_sig_models.empty else None

    best_tradeoff = results_df.sort_values(
        ['num_significant', 'aic'],
        ascending=[False, True]
    ).iloc[0]

    # Print results
    print("🏆 BEST AIC MODEL:")
    print(best_aic[['order','seasonal_order','aic','num_significant']])

    print("\n🏆 BEST BIC MODEL:")
    print(best_bic[['order','seasonal_order','bic','num_significant']])

    if best_all_sig is not None:
        print("\n🏆 BEST MODEL (ALL VARIABLES SIGNIFICANT):")
        print(best_all_sig[['order','seasonal_order','aic']])
    else:
        print("\n⚠️ No model found where ALL variables are significant")

    print("\n🏆 BEST TRADEOFF MODEL:")
    print(best_tradeoff[['order','seasonal_order','aic','num_significant']])

    return results_df, best_aic, best_bic, best_all_sig, best_tradeoff


# 🔥 RUN MODEL SELECTION (THIS WAS MISSING)
results_df, best_aic, best_bic, best_all_sig, best_tradeoff = evaluate_arima_models(y, X)

# 🔥 CHOOSE MODEL (THIS WAS MISSING)
if best_all_sig is not None:
    model_fit = best_all_sig['model_obj']
    print("\n✅ Using ALL-SIGNIFICANT model")
else:
    model_fit = best_tradeoff['model_obj']
    print("\n⚠️ Using BEST TRADEOFF model")

# 🔥 OPTIONAL: SEE TOP MODELS
print("\nTop 10 models by AIC:")
print(results_df.sort_values('aic').head(10)[
    ['order','seasonal_order','aic','bic','num_significant','significant_vars']
])

# ---------------------------------------------------------
# 6. FORECAST SETUP
# ---------------------------------------------------------
forecast_steps = 40
last_hist_date = df_history.index[-1]

forecast_dates = pd.date_range(
    start=last_hist_date,
    periods=forecast_steps + 1,
    freq='QE'
)[1:]

last_unemp = df_history['Unemployment_Rate'].iloc[-1]
last_gdp = df_history['GDP'].iloc[-1]
last_rates = df_history['Rates'].iloc[-1]

# ---------------------------------------------------------
# 7–10 REMAIN UNCHANGED
# ---------------------------------------------------------

🔍 Running FULL model evaluation...

